### Run a Small Transformer on GPU

In [2]:
import torch
import warnings
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

Device: cuda


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### Prepare Inference Input

In [3]:
text = ["This is a great demo for GPU acceleration."] * 64
inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)


### Full Precision (FP32) Inference Timing

In [4]:
import time

# Warm-up
with torch.no_grad():
    for _ in range(5):
        model(**inputs)

torch.cuda.synchronize()

start = time.time()
with torch.no_grad():
    for _ in range(50):
        model(**inputs)
torch.cuda.synchronize()
end = time.time()

fp32_time = end - start
print(f"FP32 Inference Time: {fp32_time:.3f} seconds")

FP32 Inference Time: 1.126 seconds


### Mixed Precision (FP16) Inference Timing 🚀

In [5]:
from torch.cuda.amp import autocast

torch.cuda.empty_cache()

# Warm-up
with torch.no_grad(), autocast():
    for _ in range(5):
        model(**inputs)

torch.cuda.synchronize()

start = time.time()
with torch.no_grad(), autocast():
    for _ in range(50):
        model(**inputs)
torch.cuda.synchronize()
end = time.time()

fp16_time = end - start
print(f"FP16 Inference Time: {fp16_time:.3f} seconds")


FP16 Inference Time: 0.303 seconds


### GPU Utilization

In [6]:
print("==== Memory ====")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e6:.1f} MB")

print("==== GPU SPEED SUMMARY ====")
print("GPU:", torch.cuda.get_device_name(0))
print(f"FP32 Time: {fp32_time:.3f}s")
print(f"FP16 Time: {fp16_time:.3f}s")
print(f"Speedup: {fp32_time / fp16_time:.2f}x")

==== Memory ====
Allocated: 547.1 MB
Reserved: 694.2 MB
==== GPU SPEED SUMMARY ====
GPU: Tesla T4
FP32 Time: 1.126s
FP16 Time: 0.303s
Speedup: 3.72x
